# CEFR 3-Band Classification with EBM (Explainable Boosting Machine)

A focused notebook on **EBM** - a glass-box **Generalised Additive Model** (`interpret`
library). Same 0-100 pipeline as the other notebooks, but built around EBM's biggest strength:
**every feature's contribution is exact and additive** (no post-hoc approximation).

```
features -> EBM (3-class) -> probabilities -> 0-100 score -> 2 cut-points -> 3 bands
```

**Bands:** `A1-A2` (0) < `B1` (1) < `B2-C1-C2` (2). Baseline to beat: 77%; target >=82%.

**What this notebook shows**
1. **Results** - train / test / full accuracy + confusion matrices.
2. **Feature importance** - EBM's *native* additive importance, plus permutation importance on
   **both train and test** (and full), rolled up to sections.
3. **Distribution in bins** - the 0-100 score histogram and per-band score ranges.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, classification_report)
from interpret.glassbox import ExplainableBoostingClassifier

print("interpret (EBM) loaded")

## 1. Load your data  <-- FILL THIS IN

In [ ]:
# TODO: assign your dataframe
df = None
# e.g. df = pd.read_csv("your_file.csv")

## 2. Columns  <-- FILL THIS IN

In [ ]:
FEATURE_COLS = []                 # one feature per group (8-11)

# OPTIONAL: map feature -> section, for section-level importance roll-up
FEATURE_GROUPS = {}               # e.g. {"lex_div": "Vocabulary", "mlu": "Grammar"}

ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL = "ciid", "location", "split", "cefr"
TRAIN_VALUE, TEST_VALUE = "train", "test"
META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42
BAND_MAP = {"A1": 0, "A2": 0, "B1": 1, "B2": 2, "C1": 2, "C2": 2}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

BAND_ANCHORS = np.array([0.0, 50.0, 100.0])   # score = sum_k P(band_k) * anchor_k
OPTIMIZE_METRIC = "accuracy"                  # or "balanced_accuracy"
PERM_REPEATS = 20
BASELINE_ACC, TARGET_ACC = 0.77, 0.82

## 4. Build train / test  (from the `split` column)

In [ ]:
assert df is not None and len(FEATURE_COLS) > 0, "Fill in df and FEATURE_COLS."
miss = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not miss, f"missing columns: {miss}"

def to_band(s):
    s = pd.Series(s)
    if s.dtype.kind in "iuf" and set(pd.unique(s.dropna())) <= {0, 1, 2}:
        return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    m = key.map(BAND_MAP); assert m.notna().all(), f"unmapped: {key[m.isna()].unique()}"
    return m.astype(int).to_numpy()

sp = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_df, test_df = df.loc[sp == TRAIN_VALUE].copy(), df.loc[sp == TEST_VALUE].copy()
X_train, X_test = train_df[FEATURE_COLS].astype(float), test_df[FEATURE_COLS].astype(float)
y_train, y_test = to_band(train_df[LABEL_COL]), to_band(test_df[LABEL_COL])

print(f"train/test rows: {len(X_train)}/{len(X_test)} | dropped bad flags: {(~sp.isin([TRAIN_VALUE, TEST_VALUE])).sum()}")
print("train band counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test  band counts:", dict(zip(*np.unique(y_test,  return_counts=True))))

## 5. Utilities  (0-100 score + cut-points)

In [ ]:
def proba_to_score(proba, anchors=BAND_ANCHORS):
    return np.asarray(proba) @ np.asarray(anchors, float)

def apply_cutpoints(scores, t1, t2):
    scores = np.asarray(scores)
    return np.where(scores <= t1, 0, np.where(scores <= t2, 1, 2))

def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score

def fit_cutpoints(scores, y, metric=OPTIMIZE_METRIC, n_grid=120):
    scores = np.asarray(scores); sc = _scorer(metric)
    cand = np.unique(np.percentile(scores, np.linspace(0, 100, n_grid)))
    best_v, best = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = sc(y, apply_cutpoints(scores, cand[i], cand[j]))
            if v > best_v:
                best_v, best = v, (float(cand[i]), float(cand[j]))
    return best, best_v

def band_metrics(y, pred):
    return dict(acc=accuracy_score(y, pred), bal=balanced_accuracy_score(y, pred),
                mf1=f1_score(y, pred, average="macro"),
                qwk=cohen_kappa_score(y, pred, weights="quadratic"))
print("utilities ready")

## 6. Train the EBM and get results

Direct 3-class EBM (it handles multiclass natively), then the shared pipeline:
probabilities -> expected-value 0-100 score -> 2 cut-points (tuned on train) -> bands.

In [ ]:
ebm = ExplainableBoostingClassifier(interactions=5, outer_bags=8, random_state=RANDOM_STATE)
pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", ebm)])
pipe.fit(X_train, y_train)

p_tr, p_te = pipe.predict_proba(X_train), pipe.predict_proba(X_test)
s_tr, s_te = proba_to_score(p_tr), proba_to_score(p_te)
(t1, t2), _ = fit_cutpoints(s_tr, y_train)
pred_tr, pred_te = apply_cutpoints(s_tr, t1, t2), apply_cutpoints(s_te, t1, t2)

# Beta reshaping (quantile -> Beta(5,5) bell) of the EBM 0-100 score, fitted on train
from sklearn.preprocessing import QuantileTransformer
from scipy.stats import beta as _beta
_qt = QuantileTransformer(output_distribution="uniform", n_quantiles=min(len(s_tr), 1000),
                          subsample=1000000000, random_state=RANDOM_STATE).fit(s_tr.reshape(-1, 1))
def to_bell(s, a=5.0, eps=1e-3):
    p = np.clip(_qt.transform(np.asarray(s, float).reshape(-1, 1)).ravel(), eps, 1 - eps)
    return 100.0 * _beta.ppf(p, a, a)
bell_tr, bell_te = to_bell(s_tr), to_bell(s_te)
bell_cuts = tuple(float(x) for x in to_bell(np.array([t1, t2])))

y_full = np.concatenate([y_train, y_test]); pred_full = np.concatenate([pred_tr, pred_te])
mt = band_metrics(y_test, pred_te)
flag = "PASS >=82%" if mt["acc"] >= TARGET_ACC else ("beats 77%" if mt["acc"] >= BASELINE_ACC else "below 77%")
print("=== EBM band accuracy ===")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {mt['acc']:.3f}   <-- honest  [{flag}]")
print(f"  FULL (train+test) {accuracy_score(y_full, pred_full):.3f}   (inflated - do not quote)")
print(f"  test: balanced {mt['bal']:.3f} | macroF1 {mt['mf1']:.3f} | QWK {mt['qwk']:.3f}")
print(f"  cut-points {t1:.1f} / {t2:.1f}   (baseline {BASELINE_ACC:.0%}, target {TARGET_ACC:.0%})")

### Confusion matrices - train / test / full

In [ ]:
for title, yt, yp in [("TRAIN", y_train, pred_tr), ("TEST", y_test, pred_te),
                      ("FULL (train+test)", y_full, pred_full)]:
    print(f"\n----- {title}  (accuracy {accuracy_score(yt, yp):.3f}) -----")
    display(pd.DataFrame(confusion_matrix(yt, yp, labels=[0, 1, 2]),
                         index=[f"true {b}" for b in BAND_NAMES],
                         columns=[f"pred {b}" for b in BAND_NAMES]))
print("\n----- TEST classification report -----")
print(classification_report(y_test, pred_te, target_names=BAND_NAMES))

## 7. Feature importance

### 7a. EBM native importance (exact, additive)
EBM is a glass-box GAM: each term's importance is the **mean absolute contribution** to the
prediction across the data - an exact figure, not a post-hoc approximation. Terms with `&` are
pairwise interactions.

In [ ]:
g = pipe.named_steps["model"].explain_global().data()
terms = pd.DataFrame({"term": g["names"], "importance": g["scores"]})
terms["kind"] = np.where(terms["term"].str.contains("&"), "interaction", "main effect")
terms["section"] = terms["term"].map(lambda t: FEATURE_GROUPS.get(t, t))
terms = terms.sort_values("importance", ascending=False).reset_index(drop=True)
print("EBM native term importances (mean absolute contribution)")
display(terms.round(4))

try:
    import matplotlib.pyplot as plt
    top = terms.head(12).iloc[::-1]
    plt.figure(figsize=(7, 4))
    plt.barh(top["term"], top["importance"], color="#3b6ea5")
    plt.title("EBM term importance"); plt.xlabel("mean |contribution|"); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

### 7b. Permutation importance on TRAIN and TEST
Shuffle one feature, re-run the whole pipeline (EBM -> score -> cut-points), and measure the
drop in **band accuracy**. Measured on **train, test and full**, and turned into a single
share-of-total percentage per feature (negatives floored at 0). Also rolled up to sections.

In [ ]:
def banded_predict(model, X, t1, t2):
    return apply_cutpoints(proba_to_score(model.predict_proba(X)), t1, t2)

def perm_importance(model, X, y, t1, t2, n_repeats=PERM_REPEATS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, banded_predict(model, X, t1, t2))
    recs = []
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(base - accuracy_score(y, banded_predict(model, Xp, t1, t2)))
        recs.append((col, float(np.mean(drops))))
    return pd.DataFrame(recs, columns=["feature", "imp"]).set_index("feature")["imp"], base

X_full = pd.concat([X_train, X_test])
imp = pd.DataFrame(index=FEATURE_COLS)
imp["imp_train"], b_tr = perm_importance(pipe, X_train, y_train, t1, t2)
imp["imp_test"],  b_te = perm_importance(pipe, X_test,  y_test,  t1, t2)
imp["imp_full"],  b_fu = perm_importance(pipe, X_full,  y_full,  t1, t2)

def pct(col):
    p = imp[col].clip(lower=0)
    return (100 * p / p.sum()).round(1) if p.sum() > 0 else p * 0.0
for c, tag in [("imp_train", "train"), ("imp_test", "test"), ("imp_full", "full")]:
    imp[f"pct_{tag}"] = pct(c)
imp["section"] = [FEATURE_GROUPS.get(f, f) for f in imp.index]
imp = imp.sort_values("imp_test", ascending=False)

print(f"baseline band accuracy: train {b_tr:.3f} | test {b_te:.3f} | full {b_fu:.3f}")
print("importance = drop in band accuracy when shuffled; pct_* = share of total (sums to ~100%)\n")
display(imp[["section", "imp_train", "imp_test", "imp_full", "pct_train", "pct_test", "pct_full"]].round(4))

drop = imp[imp["imp_test"] <= 0].index.tolist()
if drop:
    print("no help on test (drop candidates):", drop)

if FEATURE_GROUPS:
    print("\nSECTION-level percentage (each column sums to ~100%):")
    display(imp.groupby("section")[["pct_train", "pct_test", "pct_full"]].sum()
               .sort_values("pct_test", ascending=False).round(1))

## 8. Distribution in bins - raw vs Beta bell

The 0-100 score (full dataset): raw (U-shaped) vs the Beta-reshaped bell, with the split points
marked - the same raw-vs-Beta pair as the hierarchical notebook.

In [ ]:
s_full = np.concatenate([s_tr, s_te]); bell_full = np.concatenate([bell_tr, bell_te])
print(f"raw  split points : {t1:.1f} / {t2:.1f}")
print(f"bell split points : {bell_cuts[0]:.1f} / {bell_cuts[1]:.1f}")
print("\nbell score by band (full data):")
for b in range(N_BANDS):
    v = bell_full[pred_full == b]
    if len(v):
        print(f"  {BAND_NAMES[b]:<9} {v.min():.0f}-{v.max():.0f}  median {np.median(v):.0f}  n={len(v)}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].hist(s_full, bins=20, range=(0, 100), color="#c0553b")
    for c in (t1, t2): ax[0].axvline(c, color="k", ls="--", lw=1.5)
    ax[0].set_title(f"raw score  (splits {t1:.0f}/{t2:.0f})")
    ax[1].hist(bell_full, bins=20, range=(0, 100), color="#3b6ea5")
    for c in bell_cuts: ax[1].axvline(c, color="k", ls="--", lw=1.5)
    ax[1].set_title(f"bell after Beta  (splits {bell_cuts[0]:.0f}/{bell_cuts[1]:.0f})")
    for a in ax:
        a.set_xlim(0, 100); a.set_xlabel("0-100 score"); a.set_ylabel("learners")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 9. Export: `ebm.csv`

Per learner: id, location, split, label, and the EBM raw 0-100 score, its Beta-reshaped (bell)
score, and its predicted band. Saved to `ebm.csv`.

In [ ]:
n_tr = len(X_train)
csv = pd.DataFrame({
    ID_COL:     np.concatenate([train_df[ID_COL].values, test_df[ID_COL].values]),
    "location": np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":    ["train"] * n_tr + ["test"] * len(X_test),
    "label":    [BAND_NAMES[i] for i in y_full],
    "ebm_raw":  np.round(s_full, 2),
    "ebm_bell": np.round(bell_full, 2),
    "ebm_pred": [BAND_NAMES[i] for i in pred_full],
})
csv.to_csv("ebm.csv", index=False)
print("saved 'ebm.csv' |", len(csv), "rows |", list(csv.columns))
with pd.option_context("display.max_rows", 400, "display.max_columns", 60):
    display(csv.head(10))

## Notes

- **Why EBM:** it is the only model here whose per-feature (per-section) contributions are
  **exact and native** (section 7a) - the strongest interpretability story for the 0-100 score.
- **Importance to quote:** the permutation `pct_test` (share of real test-band accuracy). EBM's
  native importance (7a) is an internal view; permutation on test is the deliverable-level one.
- **Small test set:** treat the exact importance numbers as approximate (ordering is the signal).
- **Knobs:** `interactions` (pairwise terms), `outer_bags` (stability). Add light tuning if you
  want to push accuracy.